# Fine-Tuning

In [2]:
data = [
  "He composes songs and practices piano daily.",
  "She reads books and explores the nearby caves."
]

In [3]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMBED_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [4]:
import os
import re
import torch
import json
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader


In [5]:
save_dir = "models"

In [6]:
_ = torch.manual_seed(123)

In [7]:
def separate_dots(s):
    s = re.sub(r"\.", " . ", s)   # separate ALL punctuation
  
    return s.strip()

text_data = [separate_dots(x) for x in data]

In [8]:
file_name = "vocab.json"
with open(os.path.join(save_dir,file_name)) as f:
   vocab = json.load(f)

In [9]:
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: .
1: He
2: She
3: and
4: art
5: books
6: builds
7: caves
8: climbs
9: collaborates
10: complex
11: composes
12: creative
13: curates
14: daily
15: designs
16: digital
17: documents
18: every
19: everyday
20: exhibitions
21: experiments
22: explores
23: fairs
24: filmmakers
25: for
26: friends
27: harmonies
28: her
29: in
30: jewelry
31: local
32: maps
33: marathons
34: models
35: mountains
36: music
37: music
38: navigation
39: nearby
40: newspaper
41: novel
42: novels
43: organizes
44: participates
45: photography
46: piano
47: practices
48: projects
49: puzzles
50: reads
51: regularly
52: rhythms
53: science
54: small
55: solves
56: songs
57: soundtracks
58: stars
59: studies
60: teaches
61: the
62: trains
63: trips
64: tunes
65: using
66: weekend
67: wildlife
68: with
69: wooden
70: writes


In [10]:

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}

In [11]:

def text_to_token_ids(text, word_to_id):
    tokens = text.split()
    return [word_to_id[t] for t in tokens]

def token_ids_to_text(token_ids, id_to_word):
    words = [id_to_word[i] for i in token_ids]
    return ' '.join([id_to_word[id] for id in words])

In [12]:
class LLMDataset(Dataset):
    def __init__(self, texts, word_to_id, max_len):
        self.texts = texts
        self.word_to_id = word_to_id
        self.max_len = max_len
       
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.texts[idx], self.word_to_id)[:self.max_len+1])
        x = tokens[:-1]
        y = tokens[1:]
        return x, y

train_dataset = LLMDataset(
    text_data, 
    word_to_id, 
    CONTEXT_LEN
    )

train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False
    )
print(train_dataloader)
print(len(train_dataloader))



1


In [13]:
class Attention(nn.Module):
    def __init__(self,d_in,d_out, context_length):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out,bias=False)
        self.W_key = nn.Linear(d_in, d_out,bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length),diagonal=1))
    
    
    def forward(self, x, return_weights=False):
        _, num_tokens, _ = x.shape

        queries = self.W_query(x)  # (batch_size, seq_len, embed_dim)
        keys = self.W_key(x)    # (batch_size, seq_len, embed_dim)
        value = self.W_value(x)  # (batch_size, seq_len, embed_dim)

        attn_scores = queries @ keys.transpose(1, 2)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()
       
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)
       
        attn_weights = torch.softmax(attn_scores/(self.d_out **0.5), dim=-1)

        context_vec = attn_weights @ value

        if return_weights:
            return context_vec, attn_weights
        return context_vec


In [14]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att = Attention(
            d_in= EMBED_DIM,
            d_out= EMBED_DIM, 
            context_length= CONTEXT_LEN)
        
    def forward(self,x):
        shortcut = x

        x = self.att(x)
        x = x + shortcut
        return x

In [15]:
class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMBED_DIM)
      
        self.trf_block1 = TransformerBlock()
        self.trf_block2 = TransformerBlock()
        self.trf_block3 = TransformerBlock()
        self.trf_block4 = TransformerBlock()
        self.trf_block5 = TransformerBlock()
        self.trf_block6 = TransformerBlock()
        self.output_layer = nn.Linear(EMBED_DIM, VOCAB_SIZE)

    
    def forward(self,in_idx):
        _, seq_len = in_idx.shape

        token_emb = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len))

        x = token_emb + pos_embeds

        x = self.trf_block1(x)
        x = self.trf_block2(x)
        x = self.trf_block3(x)
        x = self.trf_block4(x)  
        x = self.trf_block5(x)
        x = self.trf_block6(x)

        logits = self.output_layer(x)
        
        return logits

In [16]:
model = GPTModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [17]:
#file_name = "parameters.bin"
#model.load_state_dict(torch.load(os.path.join(save_dir,file_name)))

file_name = "parameters.bin"
path = os.path.join(save_dir, file_name)

state_dict = torch.load(path, map_location='cpu')

fixed_state_dict = {}

for k, v in state_dict.items():
    new_key = k.replace("trm_block", "trf_block")
    fixed_state_dict[new_key] = v

missing, unexpected = model.load_state_dict(fixed_state_dict, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: ['output_layer.bias']
Unexpected keys: []


In [18]:
def train(dataloader, model, criterion, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        X = X.long()
        y = y.long()
        logits = model(X)
        loss = criterion(logits.flatten(0, 1), y.flatten())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        print(f"Batch: {batch+1}, Loss: {loss:>7f}")
        
  

In [19]:
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}")
    train(train_dataloader, model, criterion, optimizer)
    print("-------------------------------")
print("Training done!")    

Epoch 1
Batch: 1, Loss: 8.337070
-------------------------------
Epoch 2
Batch: 1, Loss: 7.186207
-------------------------------
Epoch 3
Batch: 1, Loss: 6.316023
-------------------------------
Epoch 4
Batch: 1, Loss: 5.716542
-------------------------------
Epoch 5
Batch: 1, Loss: 5.169549
-------------------------------
Epoch 6
Batch: 1, Loss: 4.648851
-------------------------------
Epoch 7
Batch: 1, Loss: 4.148213
-------------------------------
Epoch 8
Batch: 1, Loss: 3.684642
-------------------------------
Epoch 9
Batch: 1, Loss: 3.303064
-------------------------------
Epoch 10
Batch: 1, Loss: 2.974054
-------------------------------
Epoch 11
Batch: 1, Loss: 2.681694
-------------------------------
Epoch 12
Batch: 1, Loss: 2.423576
-------------------------------
Epoch 13
Batch: 1, Loss: 2.198474
-------------------------------
Epoch 14
Batch: 1, Loss: 1.993086
-------------------------------
Epoch 15
Batch: 1, Loss: 1.798699
-------------------------------
Epoch 16
Batch: 1, 